# 03 — Query Ingested Data & Compare Approaches

Verifies that Zerobus-pushed data arrived and runs the **same analytics**
as Part 1 for a direct comparison.

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "streaming_demo")

ZEROBUS_TABLE = f"{UC_CATALOG}.{UC_SCHEMA}.slot_events_zerobus"

## Verify data arrived

In [2]:
zb_df = spark.table(ZEROBUS_TABLE)
print(f"Zerobus table row count: {zb_df.count()}")
zb_df.show(10, truncate=False)

Zerobus table row count: 500
+------------------------------------+----------+------------+--------------------+---------+----------+----------+-------------------+
|event_id                            |machine_id|casino_floor|game_type           |player_id|bet_amount|win_amount|event_timestamp    |
+------------------------------------+----------+------------+--------------------+---------+----------+----------+-------------------+
|e65db416-4a1c-4844-ad10-51adce8d564b|MCH-0009  |Main Floor  |Slots               |PLR-31417|1.49      |0.0       |2026-03-25T10:00:14|
|10f06b73-71cc-49aa-b6fd-0586b09ebb5a|MCH-0006  |High Roller |Keno                |PLR-46463|2.65      |0.0       |2026-03-25T10:00:01|
|50b64800-cad7-4ba9-acf4-2106fc98e5ac|MCH-0035  |Penny Lane  |Keno                |PLR-87236|12.44     |19.58     |2026-03-25T10:00:44|
|f1dd735b-096e-4739-a1f4-728f5f48ed74|MCH-0011  |Penny Lane  |Electronic Blackjack|PLR-55082|593.37    |1198.31   |2026-03-25T10:00:51|
|a30eeac6-55d3-456f

## Revenue by Casino Floor

Same query as Part 1, notebook 04 — but on the Zerobus-ingested table.

In [3]:
spark.sql(f"""
    SELECT
        casino_floor,
        COUNT(*) AS total_plays,
        ROUND(SUM(bet_amount), 2) AS total_bets,
        ROUND(SUM(win_amount), 2) AS total_payouts,
        ROUND(SUM(bet_amount) - SUM(win_amount), 2) AS house_revenue
    FROM {ZEROBUS_TABLE}
    GROUP BY casino_floor
    ORDER BY house_revenue DESC
""").show(truncate=False)

+------------+-----------+----------+-------------+-------------+
|casino_floor|total_plays|total_bets|total_payouts|house_revenue|
+------------+-----------+----------+-------------+-------------+
|VIP Lounge  |93         |12769.03  |8084.36      |4684.67      |
|Main Floor  |99         |13989.51  |15662.27     |-1672.76     |
|Sports Zone |103        |15540.82  |17665.17     |-2124.35     |
|Penny Lane  |103        |16960.18  |21633.78     |-4673.6      |
|High Roller |102        |15213.66  |32613.89     |-17400.23    |
+------------+-----------+----------+-------------+-------------+



## Top 10 Machines by Play Volume

In [4]:
spark.sql(f"""
    SELECT
        machine_id,
        casino_floor,
        game_type,
        COUNT(*) AS total_plays,
        ROUND(SUM(bet_amount), 2) AS total_wagered
    FROM {ZEROBUS_TABLE}
    GROUP BY machine_id, casino_floor, game_type
    ORDER BY total_plays DESC
    LIMIT 10
""").show(truncate=False)

+----------+------------+--------------------+-----------+-------------+
|machine_id|casino_floor|game_type           |total_plays|total_wagered|
+----------+------------+--------------------+-----------+-------------+
|MCH-0030  |Main Floor  |Electronic Roulette |3          |538.95       |
|MCH-0042  |Main Floor  |Electronic Blackjack|3          |1722.18      |
|MCH-0023  |Sports Zone |Electronic Roulette |3          |1151.77      |
|MCH-0014  |High Roller |Keno                |3          |8.14         |
|MCH-0004  |Sports Zone |Electronic Blackjack|3          |649.01       |
|MCH-0035  |VIP Lounge  |Keno                |3          |21.86        |
|MCH-0029  |Main Floor  |Keno                |3          |38.6         |
|MCH-0050  |Sports Zone |Electronic Blackjack|3          |1378.7       |
|MCH-0028  |Sports Zone |Electronic Roulette |3          |600.87       |
|MCH-0050  |Main Floor  |Slots               |3          |75.27        |
+----------+------------+--------------------+-----

## Win/Loss Distribution by Game Type

In [5]:
spark.sql(f"""
    SELECT
        game_type,
        COUNT(*) AS total_plays,
        SUM(CASE WHEN win_amount > 0 THEN 1 ELSE 0 END) AS wins,
        SUM(CASE WHEN win_amount = 0 THEN 1 ELSE 0 END) AS losses,
        ROUND(SUM(CASE WHEN win_amount > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS win_pct,
        ROUND(AVG(bet_amount), 2) AS avg_bet,
        ROUND(AVG(CASE WHEN win_amount > 0 THEN win_amount END), 2) AS avg_win_payout
    FROM {ZEROBUS_TABLE}
    GROUP BY game_type
    ORDER BY total_plays DESC
""").show(truncate=False)

+--------------------+-----------+----+------+-------+-------+--------------+
|game_type           |total_plays|wins|losses|win_pct|avg_bet|avg_win_payout|
+--------------------+-----------+----+------+-------+-------+--------------+
|Keno                |112        |54  |58    |48.2   |9.97   |34.58         |
|Electronic Roulette |101        |44  |57    |43.6   |270.19 |509.12        |
|Slots               |100        |39  |61    |39.0   |23.34  |60.9          |
|Video Poker         |99         |43  |56    |43.4   |48.48  |165.19        |
|Electronic Blackjack|88         |42  |46    |47.7   |442.43 |1474.11       |
+--------------------+-----------+----+------+-------+-------+--------------+



## Approach Comparison: Structured Streaming vs. Zerobus Ingest

| | **Structured Streaming (Part 1)** | **Zerobus Ingest (Part 2)** |
|---|---|---|
| **How data arrives** | Files land in cloud storage (or Kafka topic) | Pushed directly from source to Lakehouse |
| **Infrastructure needed** | File storage + Auto Loader (or Kafka broker) | Just the Zerobus endpoint — no message bus |
| **Latency** | Seconds to minutes (depends on trigger interval) | Sub-5 seconds end-to-end |
| **Processing model** | Full Spark — transforms, joins, aggregations in the stream | Ingestion only — transform after landing |
| **Best for** | Complex ETL pipelines, multi-step transforms | High-volume event push (IoT, telemetry, clickstream) |
| **Kafka relationship** | Reads from Kafka as a source | Replaces Kafka entirely for single-destination flows |

**Bottom line:** If your data needs complex in-flight transformation, Structured Streaming
is the right tool. If you're pushing events from devices/apps and your sole destination
is the Lakehouse, Zerobus removes an entire infrastructure layer.